# Lab dl_06 · A byte-level BPE tokenizer

**Goal.** Implement byte-pair encoding the way GPT-2 does it at the byte level: train merges on text, encode, decode, and
compare with OpenAI's `tiktoken` GPT-2 tokenizer.

### Theory recap
Start from the UTF-8 **bytes** of the text (256 base tokens — every string is representable, no unknown token). Repeat:
1. count all adjacent pairs (overlapping: `aaa` contains `(a,a)` twice);
2. take the most frequent pair (ties: the one that **occurs first** in the sequence);
3. replace its occurrences left to right, non-overlapping, by a new id $256 + k$, and record the merge.

**Encoding** new text replays the merges in the order they were learned: repeatedly find the adjacent pair with the lowest
merge index and merge it, until no learned pair is present. **Decoding** concatenates each id's bytes and decodes UTF-8.
Vocabulary size $= 256 + \#\text{merges} + \#\text{special tokens}$ (GPT-2: $256 + 50000 + 1 = 50257$).

**How this works.** Fill in every cell marked `# TODO` (they currently `raise NotImplementedError`). Run the notebook top to bottom with the working directory set to `labs/`. The last cell runs hidden checks on your functions and records the result in the Learning Center. Do the paper homework for the unlocking skill first — the lab assumes you can do the math by hand.

*This lab is original course material, inspired by Andrej Karpathy's open-source teaching code ([micrograd](https://github.com/karpathy/micrograd), [nn-zero-to-hero](https://github.com/karpathy/nn-zero-to-hero), [nanoGPT](https://github.com/karpathy/nanoGPT), [build-nanogpt](https://github.com/karpathy/build-nanogpt), [minbpe](https://github.com/karpathy/minbpe)). Watching the corresponding lecture after finishing is highly recommended.*

In [ ]:
import os
from pathlib import Path

text = Path("data/tinyshakespeare.txt").read_text(encoding="utf-8")
print(len(text), "characters,", len(text.encode("utf-8")), "bytes")

## 1. Pair statistics and merging

In [ ]:
def get_stats(ids, counts=None):
    # TODO: dict {(a, b): count} over adjacent pairs; insertion order = first occurrence
    raise NotImplementedError


def merge(ids, pair, idx):
    # TODO: replace non-overlapping occurrences of `pair` (left to right) by idx
    raise NotImplementedError


print(get_stats([1, 2, 3, 1, 2, 2, 2]))
print(merge([5, 5, 5, 5, 5], (5, 5), 7))

## 2. The tokenizer
`BasicTokenizer.train(text, vocab_size)` fills `self.merges` (dict `(int, int) -> int`, insertion order = training order) and `self.vocab` (dict `int -> bytes`). `encode(text) -> list[int]`, `decode(ids) -> str` (use `errors="replace"`).

In [ ]:
class BasicTokenizer:
    def __init__(self):
        self.merges = {}
        self.vocab = {i: bytes([i]) for i in range(256)}

    def train(self, text, vocab_size, verbose=False):
        # TODO
        raise NotImplementedError

    def encode(self, text):
        # TODO
        raise NotImplementedError

    def decode(self, ids):
        # TODO
        raise NotImplementedError

In [ ]:
import time
t0 = time.time()
tok = BasicTokenizer().train(text[:100_000], 256 + 300)
print(f"trained 300 merges in {time.time() - t0:.1f}s")
for (a, b), idx in list(tok.merges.items())[:15]:
    print(idx, repr(tok.vocab[idx]))
sample = text[200_000:201_000]
ids = tok.encode(sample)
assert tok.decode(ids) == sample
print(f"{len(sample.encode())} bytes -> {len(ids)} tokens  (compression {len(sample.encode()) / len(ids):.2f}x)")

## 3. Compare with tiktoken's GPT-2 encoding
The GPT-2 vocabulary files were cached in `data/tiktoken_cache/` so this works offline. GPT-2 also pre-splits text with a regex (so merges never cross word/punctuation boundaries) — notice the effect on tokens like `'s` and spaces.

In [ ]:
os.environ.setdefault("TIKTOKEN_CACHE_DIR", os.path.abspath("data/tiktoken_cache"))
try:
    import tiktoken
    enc = tiktoken.get_encoding("gpt2")
    gids = enc.encode(sample)
    print(f"tiktoken gpt2: {len(gids)} tokens (compression {len(sample.encode()) / len(gids):.2f}x), vocab {enc.n_vocab}")
    print("ours :", [tok.decode([i]) for i in tok.encode("The king's men, hearing this,")])
    print("gpt2 :", [enc.decode([i]) for i in enc.encode("The king's men, hearing this,")])
except Exception as exc:  # offline without cache, or tiktoken missing
    print("tiktoken unavailable, skipping comparison:", exc)

### Reflect (on paper)
1. Train with 300 vs. 3000 merges (on a 100 kB slice). How does compression change? What does a larger vocabulary cost in the model (embedding and output layer)?
2. Why does GPT-2 work on bytes rather than Unicode characters? How many initial tokens does `東京🙂` produce?
3. Why can a tokenizer trained on English compress code or non-English text poorly, and what does that mean for the context length and cost of those users?

## Grade
Runs the hidden checks (fast, small inputs, fixed seeds) and reports to the local app.

In [ ]:
from lc_client import grade
grade("dl_06_bpe", globals())